In [0]:
#Streaming data for preprocessing

display(dbutils.fs.ls("dbfs:/Volumes/workspace/default/telegram_project"))

In [0]:
# create folder raw
dbutils.fs.mkdirs("dbfs:/Volumes/workspace/default/telegram_project/raw")

# create folder processed
dbutils.fs.mkdirs("dbfs:/Volumes/workspace/default/telegram_project/processed")

In [0]:
from huggingface_hub import login 
from dotenv import load_dotenv
import os 
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
login(HF_TOKEN)



In [0]:
pip install datasets 

In [0]:
from huggingface_hub import HfApi
import os
from dotenv import load_dotenv
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

api = HfApi(token=hf_token)

repo_id = "leonardoblas/us_election_2024_telegram_distilled"

files = api.list_repo_files(
    repo_id=repo_id,
    repo_type="dataset"
)

ch14_files = [
    f for f in files
    if f.startswith("channels_14/") and f.endswith(".db")
]

print("=== Found channels_14 .db files ===")
for f in ch14_files:
    print(f)

print("\nTotal:", len(ch14_files))

In [0]:
import os
import math
import sqlite3
import shutil
import pandas as pd

from huggingface_hub import HfApi, hf_hub_download

In [0]:
def pick_df_col(df, col, default=None):
    if col in df.columns:
        return df[col]
    return pd.Series([default] * len(df))

def safe_int_series(s, default=0):
    return pd.to_numeric(s, errors="coerce").fillna(default).astype("Int64")

def safe_float_series(s, default=0.0):
    return pd.to_numeric(s, errors="coerce").fillna(default).astype(float)

def safe_str_series(s, default=""):
    return s.fillna(default).astype(str)

def normalize_chunk(df, channel_name, source_db):
    expected_cols = [
        "message_id", "content", "language", "from_id", "post_author",
        "edit_hide", "noforwards", "scheduled", "pinned",
        "views", "forwards", "replies", "date", "edit_date", "ttl",
        "restriction_reasons", "forward", "reply", "reply_source",
        "reply_scheduled", "chain_from_id", "chain_from_name",
        "chain_post_author", "chain_imported", "chain_date",
        "invoice", "media_price", "contact", "location",
        "web_preview", "story", "photo", "video", "round", "voice",
        "other_media", "media_ttl", "buttons", "emails", "phones",
        "hashtags", "cashtags", "cards", "urls", "cleaner_urls",
        "domains", "identifiers", "political", "toxicity",
        "severe_toxicity", "identity_attack", "insult", "profanity",
        "threat"
    ]

    for col in expected_cols:
        if col not in df.columns:
            df[col] = None

    int_cols = [
        "message_id", "edit_hide", "noforwards", "scheduled", "pinned",
        "views", "forwards", "replies", "ttl", "forward", "reply",
        "reply_scheduled", "chain_imported", "media_price",
        "web_preview", "story", "photo", "video", "round", "voice",
        "other_media", "media_ttl", "buttons", "political"
    ]

    float_cols = [
        "toxicity", "severe_toxicity", "identity_attack",
        "insult", "profanity", "threat"
    ]

    str_cols = [
        "content", "language", "from_id", "post_author", "date", "edit_date",
        "reply_source", "chain_from_id", "chain_from_name", "chain_post_author",
        "chain_date", "invoice", "contact", "location", "emails", "phones",
        "hashtags", "cashtags", "cards", "urls", "cleaner_urls",
        "domains", "identifiers", "restriction_reasons"
    ]

    for col in int_cols:
        df[col] = safe_int_series(df[col], default=0)

    for col in float_cols:
        df[col] = safe_float_series(df[col], default=0.0)

    for col in str_cols:
        df[col] = safe_str_series(df[col], default="")

    df["channel_name"] = channel_name
    df["source_db"] = source_db

    return df

In [0]:
def filter_chunk(df):
    has_content = df["content"].str.strip().str.len() > 0

    has_network_signal = (
        (df["forward"] > 0) |
        (df["forwards"] > 0) |
        (df["views"] > 0) |
        (df["replies"] > 0) |
        (df["reply"] > 0) |
        (df["chain_from_id"].str.strip().str.len() > 0) |
        (df["reply_source"].str.strip().str.len() > 0)
    )

    has_media = (
        (df["photo"] > 0) |
        (df["video"] > 0) |
        (df["round"] > 0) |
        (df["voice"] > 0) |
        (df["story"] > 0) |
        (df["other_media"] > 0) |
        (df["web_preview"] > 0) |
        (df["buttons"] > 0)
    )

    keep_mask = has_content | has_network_signal | has_media
    return df[keep_mask].copy()

In [0]:
HF_TOKEN = os.getenv("HF_TOKEN")
repo_id = "leonardoblas/us_election_2024_telegram_distilled"

api = HfApi(token=HF_TOKEN)

files = api.list_repo_files(
    repo_id=repo_id,
    repo_type="dataset"
)

db_files = [f for f in files if f.endswith(".db")]

print("Total .db files:", len(db_files))
print(db_files[42580:])

In [0]:
target_db_files = [
    f for f in db_files
    if f.startswith("channels_15/")
]
print("Target .db files:", len(target_db_files))
print(target_db_files[:20])

In [0]:
output_dir = "/Volumes/workspace/default/telegram_project/processed/messages_clean_parts_from_db_channels115"
os.makedirs(output_dir, exist_ok=True)

tmp_dir = "/local_disk0/tmp_hf_db"
import tempfile
tmp_dir = tempfile.mkdtemp()


In [0]:
rows_seen_total = 0
rows_kept_total = 0
part_idx = 0


for file_idx, repo_file in enumerate(target_db_files[:], start=1):
    channel_name = repo_file.split("/")[0]
    db_filename = os.path.basename(repo_file)

    print(f"\n=== [{file_idx}/{len(target_db_files)}] Processing {repo_file} ===")

    try:
        # tải 1 file từ HF về cache local
        local_cached_path = hf_hub_download(
            repo_id=repo_id,
            filename=repo_file,
            repo_type="dataset",
            token=HF_TOKEN
        )

        # copy sang tmp riêng để dễ quản lý
        local_db_path = os.path.join(tmp_dir, db_filename)
        shutil.copy2(local_cached_path, local_db_path)

        # mở sqlite
        conn = sqlite3.connect(local_db_path)

        # kiểm tra có bảng messages không
        tables_df = pd.read_sql("""
            SELECT name
            FROM sqlite_master
            WHERE type='table'
        """, conn)

        table_names = set(tables_df["name"].tolist())
        if "messages" not in table_names:
            print(f"Skip {repo_file}: no 'messages' table")
            conn.close()
            os.remove(local_db_path)
            continue

        # đọc theo chunk để đỡ RAM
        query = "SELECT * FROM messages"
        chunk_iter = pd.read_sql_query(query, conn, chunksize=50000)

        file_seen = 0
        file_kept = 0

        for chunk_no, chunk in enumerate(chunk_iter, start=1):
            original_n = len(chunk)
            file_seen += original_n
            rows_seen_total += original_n

            chunk = normalize_chunk(chunk, channel_name=channel_name, source_db=repo_file)
            chunk_keep = filter_chunk(chunk)

            kept_n = len(chunk_keep)
            file_kept += kept_n
            rows_kept_total += kept_n

            if kept_n > 0:
                part_path = os.path.join(output_dir, f"part_{part_idx:06d}.parquet")
                chunk_keep.to_parquet(part_path, index=False)
                print(
                    f"  chunk={chunk_no} | seen={original_n} | kept={kept_n} | "
                    f"saved={part_path}"
                )
                part_idx += 1
            else:
                print(f"  chunk={chunk_no} | seen={original_n} | kept=0")

        conn.close()

        # xóa file local sau khi xử lý xong
        os.remove(local_db_path)

        print(
            f"Finished {repo_file} | file_seen={file_seen} | file_kept={file_kept} | "
            f"rows_seen_total={rows_seen_total} | rows_kept_total={rows_kept_total}"
        )

    except Exception as e:
        print(f"ERROR processing {repo_file}: {e}")
        try:
            if os.path.exists(local_db_path):
                os.remove(local_db_path)
        except:
            pass
        continue

print("\n=== DONE ===")
print("rows_seen_total =", rows_seen_total)
print("rows_kept_total =", rows_kept_total)
print("output_dir =", output_dir)

In [0]:
df_spark = spark.read.parquet("/Volumes/workspace/default/telegram_project/processed/messages_clean_parts_from_db_channels115/")
print("Total kept rows:", df_spark.count())
display(df_spark.limit(10))

In [0]:
from pyspark.sql import functions as F

df = spark.read.parquet("/Volumes/workspace/default/telegram_project/processed/messages_clean_parts_from_db_channels115/")

df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.length(F.col("content")) > 0, 1).otherwise(0)).alias("rows_with_content"),
    F.sum(F.when(F.col("forward") > 0, 1).otherwise(0)).alias("rows_forward"),
    F.sum(F.when(F.col("forwards") > 0, 1).otherwise(0)).alias("rows_with_forwards"),
    F.sum(F.when(F.col("views") > 0, 1).otherwise(0)).alias("rows_with_views"),
    F.sum(F.when(F.col("photo") > 0, 1).otherwise(0)).alias("rows_photo"),
    F.sum(F.when(F.col("video") > 0, 1).otherwise(0)).alias("rows_video"),
    F.sum(F.when(F.col("other_media") > 0, 1).otherwise(0)).alias("rows_other_media")
).show()

In [0]:
df_part = spark.read.parquet(
    "dbfs:/Volumes/workspace/default/telegram_project/processed/messages_clean_parts_from_db_channels10/part_005749.parquet"
)

df_part.show(truncate=False)

In [0]:
df = spark.read.parquet(".../channel15/")

df_dedup = df.dropDuplicates(["message_id"])  # hoặc id duy nhất của bạn